# STELLAR Demo - Getting Started

This tutorial demonstrates how to install STELLAR and test the installation by sending malicious user inputs to a standalone LLM.


<div style="background-color:#fff8d6; border:1px solid #e6c200; border-radius:6px; padding:12px;">
Requirements
    
 - **LLMs:** Configure the model endpoint and API key when using Azure-hosted models. For local execution, install the required models through `ollama`.  
 - **Operating System:** The provided scripts and notebooks have been tested on Linux-based systems.  
 - **Hardware:** No high-end GPU is required when using cloud-hosted models. Example experiments use the 8B `dolphin3` model locally. If local inference is not feasible on the available hardware, cloud models can be used instead.  
 - **Software Environment:** Python 3.11.8 environment with Jupyter Notebook installed, together with `ollama` for local model serving.

 </div>

## Installation

First install `Python 3.11` based on your OS.
Then install the requirements:

In [ ]:
!pip install -r ../requirements.txt

## LLM configuration

Use `.env` to configure the Azure OpenAI endpoint API key in case you are using cloud models.

To run local models install [ollama](https://ollama.com/download) and pull below the required model using `ollama pull model-name`

## Test Installation

To verify if installation was successful, run the following code. It will start a small test experiment for the safety usecase. You can find details of this usecase in [this notebook](03_safety.ipynb).

You can test the installation using local and cloud-based LLM models.

To run with local models deployed with ollama, pull `dolphin3` first:

In [ ]:
!ollama pull dolphin3

Run the test script to generate malicious utterances and execute with LLM. Note, that depending on the hardware single test exeuction can take multiple seconds to minutes.

In [ ]:
!cd .. && bash scripts/test_safety_local.sh

To evaluate with a cloud model (GPT-4O-MINI) run the following (requires setup of AzureOpenAI endpoint and API_KEY in previous steps):

In [ ]:
!cd .. && bash scripts/test_safety_cloud.sh

## Results Output

STELLAR generates several result outputs, below are some outputs visualized. Select first which type of models you employed previously:

In [5]:
model_type = "local" # "cloud"

In [ ]:
import os
from pathlib import Path

base = Path("..") / "results" / "tests" / "safety"
search_root = base / ("cloud" if model_type == "cloud" else "local")

# Recursively find all directories containing all_utterances.json
run_dirs = sorted(
    [p.parent for p in search_root.rglob("all_utterances.json")],
    key=os.path.getmtime,
)

if not run_dirs:
    raise FileNotFoundError(f"No results found under {search_root}")

latest_run = run_dirs[-1]
print(f"Found {len(run_dirs)} run(s). Latest: {latest_run}")

You can observe the content of this folder manually or run the next cell to inspect the generated test cases. Each test case is an independently generated prompt sent to the model under test, along with the model's response and its evaluated safety score.

In [ ]:
import json
import pandas as pd

# --- Load utterances ---
with open(latest_run / "all_utterances.json") as f:
    all_utterances = json.load(f)

print(f"Total utterances generated: {len(all_utterances)}\n")
print("=" * 60)
print("Sample generated test cases:")
print("Each entry is an independent test case: generated prompt → model response → safety score")
print("=" * 60)
for i, entry in enumerate(all_utterances[:5]):
    u = entry["utterance"]
    print(f"\n[{i+1}] Prompt:   {u['question']}")
    print(f"     Response: {u['answer'][:150]}{'...' if len(u['answer']) > 150 else ''}")

# --- Load and display critical test cases ---
with open(latest_run / "all_critical_utterances.json") as f:
    critical_utterances = json.load(f)

print("\n\n" + "=" * 60)
print(f"Critical (unsafe) test cases found: {len(critical_utterances)}")
print("=" * 60)
if critical_utterances:
    for i, entry in enumerate(critical_utterances[:5]):
        u = entry["utterance"]
        print(f"\n[{i+1}] Prompt:   {u['question']}")
        print(f"     Response: {u['answer'][:150]}{'...' if len(u['answer']) > 150 else ''}")
else:
    print("No critical test cases were found in this run.")

# --- Load calculation properties ---
calc_props = pd.read_csv(latest_run / "calculation_properties.csv")

print("\n\n" + "=" * 60)
print("Calculation Properties:")
print("=" * 60)
for _, row in calc_props.iterrows():
    print(f"  {row['Attribute']:30s} : {row['Value']}")